# Evaluation Notebook

This file performs evaluation of the diffusion-based predictor, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the default model directory.
# model_dir = "/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-ensemble3/wandb/run-20250828_140541-nznhbsh7/files"
# model_dir = "/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-obsprob0.5-normalizedObs-wcg0.1/wandb/run-20260306_175250-63u0t05d/files"

# Set the output file.
output_file = "output_toy_problem_uq.zarr"

# Set the output file automatically.
# output_file = "./diffuser_kf_comparison_drift_" + model_dir.split("/")[-2] + ".zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main

import os
import yaml
import time
from copy import deepcopy
from multiprocessing import Process
os.environ["WANDB__SERVICE_WAIT"] = "300"

In [3]:
def convert_args(argsdict):
    # Convert arguments to the appropriate format.
    arglist = []
    for key, value in argsdict.items():
        if type(value) == dict and "value" in value and not key.startswith("_"):
            v = value["value"]
            if v == None:
                continue
            if type(v) == bool:
                if v:
                    arglist.append(f"--{key}")
                else:
                    arglist.append(f"--no-{key}")
            elif type(v) == list:
                if len(v) == 0:
                    continue
                arglist.append(f"--{key}")
                arglist.extend([str(i) for i in v])
            else:
                arglist.append(f"--{key}")
                arglist.append(str(v))
    return arglist


def wrap_arg_val(val):
    return {"value": val}


def load_args(model_dir):
    # Load arguments from the config file.
    config_file = os.path.join(model_dir, "config.yaml")
    args = yaml.load(open(config_file), Loader=yaml.FullLoader)

    # Set required eval-specific arguments. Do not change these!
    args["use_wandb"] = wrap_arg_val(False)
    args["use_render"] = wrap_arg_val(False)
    args["use_eval"] = wrap_arg_val(True)
    args["model_dir"] = wrap_arg_val(model_dir)
    args["eval_output_file"] = wrap_arg_val(output_file)


    # Set default arguments for evaluation. Feel free to change these!
    args["cuda"] = wrap_arg_val(True)
    args["cuda_idx"] = wrap_arg_val(5)
    args["cuda_idx_predictor"] = args["cuda_idx"]
    args["eval_episodes"] = wrap_arg_val(3)
    args["episode_length"] = wrap_arg_val(200)
    args["seed"] = wrap_arg_val(1)
    args["random_start_positions"] = wrap_arg_val(False)

    # args["disturbance_drift_magnitude"] = wrap_arg_val(0.5)

    return args

# Load arguments for the default model.
# args = load_args(model_dir)

# Evaluation

In [4]:
# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")
    # print("Output file already exists. Overwriting evaluation.")

else:
# if True:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)
    
    # obs_probabilities = np.arange(0.0, 1.0, 0.01).tolist()
    obs_probabilities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

    # Add diffuser series.
    for obs_prob in obs_probabilities:
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-uqEst/wandb/run-20260319_174150-mq3zy0j0/files")
        args["eval_series"] = wrap_arg_val(args["experiment_name"]['value'])
        args["experiment_name"] = wrap_arg_val(f"uqEst - obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["diffusion_autoregression_steps"] = wrap_arg_val(1)
        args["random_start_positions"] = wrap_arg_val(False)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-uqEnsemble3/wandb/run-20260321_195531-3dw79p9w/files")
        args["eval_series"] = wrap_arg_val(args["experiment_name"]['value'])
        args["experiment_name"] = wrap_arg_val(f"uqEnsemble3 - obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        args["diffusion_autoregression_steps"] = wrap_arg_val(1)
        args["random_start_positions"] = wrap_arg_val(False)
        start_process(args)
    
    # # Add KF series.
    # for obs_prob in obs_probabilities:
    #     args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-obsprob0.5-normalizedObs-wcg0.1/wandb/run-20260306_175250-63u0t05d/files")

    #     series_name = args["experiment_name"]['value']

    #     args["eval_series"] = wrap_arg_val(f"KF - {series_name}")
    #     args["observation_probability"] = wrap_arg_val(obs_prob)
    #     args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
    #     args["prediction_history_window"] = wrap_arg_val(1)
    #     args["prediction_ensemble_size"] = wrap_arg_val(1)
    #     args["algorithm_class"] = wrap_arg_val("")
    #     args["policy_class"] = wrap_arg_val("qmas.baseline_kf.policy.QmasPolicy")
    #     args["prediction_history_include_actions"] = wrap_arg_val(True)
    #     start_process(args)

        # argscopy = deepcopy(args)
        # argscopy["eval_series"] = wrap_arg_val(f"KF - RS - {series_name}")
        # argscopy["random_start_positions"] = wrap_arg_val(True)
        # start_process(argscopy)

    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    MAX_PROCESSES = 30
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.


/data/anthony/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '200', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-observation_mask', '--no-save_checkpoints', '--no-share_reward', '--no-threaded_training', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.



/data/anthony/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '200', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-observation_mask', '--no-save_checkpoints', '--no-share_reward', '--no-threaded_training', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")
/data/anthony/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '200', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-observation_mask', '--no-save_checkpoints', '--no-share_reward', '--no-threaded_training', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.



...

Pettingzoo arguments validated: base
Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments validation function.
Pettingzoo arguments validated: toy_problem.toy_problem_v1.parallel_env
PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.
PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': 50, 'num_agents': 1, 'num_adversaries': 1, 'num_dimensions': 2, 'world_size': 60.0, 'random_start_positions': False, 'state_per_agent': False, 'observation_probability': 0.3, 'disturbance_drift_magnitude': 0.0, 'adversary_dynamics': 'random', 'lorenz_sigma': 10.0, 'lorenz_rho': 28.0, 'lorenz_beta': 2.6666666666666665}


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/t

R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


R_Critic: Use CNN: False, Use MLP: True
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 9.
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
Creating uncertainty bounds estimator for predictor.
Initialized QmasAlgorithm with 1 predictors. Threaded training: False
Buffer initialized with episode length 200
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Critic: Use CNN: False, Use MLP: True
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 9.
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use Enco

# Analysis

In [5]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='toy-problem-uqEnsemble3', nodes=(No…

In [6]:
import plotly.graph_objects as go
import numpy as np

# Data for plotting
categories = {
    'tracking_error_mean': {
        'title': 'Tracking Error',
        'ylabel': 'Tracking Error (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['tracking_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_trajectory_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_uncertainty_mean': {
        'title': 'Prediction Uncertainty',
        'ylabel': 'Prediction Uncertainty (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_uncertainty_mean']) for experiment in output[series]]
                for series in output
        }
    },
    # 'prediction_uncertainty_error_correlation': {
    #     'title': 'Correlation between Prediction Uncertainty and Prediction Error',
    #     'ylabel': 'Correlation Coefficient',
    #     'xlabel': 'Observation Probability',
    #     # Average over all episodes and all timesteps.
    #     'y': {
    #         series: [np.mean(output[series][experiment]['prediction_uncertainty_error_correlation']) for experiment in output[series]]
    #             for series in output
    #     }
    # },
    'prediction_uncertainty_gap_mean': {
        'title': 'Prediction Uncertainty Gap',
        'ylabel': 'Prediction Uncertainty Gap (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_uncertainty_gap_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_uncertainty_gap_min': {
        'title': 'Prediction Uncertainty Gap (Min)',
        'ylabel': 'Prediction Uncertainty Gap Min (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_uncertainty_gap_min']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_uncertainty_gap_max': {
        'title': 'Prediction Uncertainty Gap (Max)',
        'ylabel': 'Prediction Uncertainty Gap Max (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['prediction_uncertainty_gap_max']) for experiment in output[series]]
                for series in output
        }
    }
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

for category in categories:
    fig = go.Figure()
    x = categories[category].get('x', None)
    
    if type(categories[category]['y']) == dict:
        for series, values in categories[category]['y'].items():
            print(f"{category} - {series}: {values}")
            x_local = x
            if x_local is None:
                x_local = np.linspace(0, 1, len(values))
            fig.add_trace(go.Scatter(
                x=x_local,
                y=values,
                mode='lines+markers',
                name=series,
                hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
            ))
    else:
        values = categories[category]['y']
        x_local = x
        if x_local is None:
            x_local = np.linspace(0, 1, len(values))
        fig.add_trace(go.Scatter(
            x=x_local,
            y=values,
            mode='lines+markers',
            hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
        ))
    
    title = categories[category]['title']
    xlabel = 'Scenario' if "xlabel" not in categories[category] else categories[category]['xlabel']
    ylabel = f"Timesteps out of {timesteps}" if "ylabel" not in categories[category] else categories[category]['ylabel']
    
    fig.update_layout(
        # title=f"{title}: avg. over {args['eval_episodes']['value']} episodes, seed {args['seed']['value']}",
        title=f"{title}",
        xaxis_title=xlabel,
        yaxis_title=ylabel,
        hovermode='closest',
        showlegend=True,
        autosize=True,
        height=600,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(
            orientation='h',
            yanchor='top',
            y=-0.2,
            xanchor='center',
            x=0.5
        ),
    )
    
    fig.show()

tracking_error_mean - toy-problem-uqEnsemble3: [3.4437757, 1.202988, 0.9290499, 0.5821757, 0.5547781, 0.4910109, 0.4403326, 0.36255783, 0.3232098, 0.30937394, 0.25057736]
tracking_error_mean - toy-problem-uqEst: [3.6434805, 1.9603622, 0.72720593, 0.57230115, 0.60292715, 0.48970127, 0.44827977, 0.36328363, 0.3619318, 0.32972825, 0.24120875]


prediction_error_mean - toy-problem-uqEnsemble3: [0.5740837, 0.19884971, 0.10756772, 0.061792273, 0.046301212, 0.0343309, 0.023226352, 0.012725299, 0.008138966, 0.0041428274, 2.7759288e-08]
prediction_error_mean - toy-problem-uqEst: [0.5492846, 0.31105453, 0.093794845, 0.06263423, 0.051682133, 0.036731288, 0.027640391, 0.015631959, 0.010530432, 0.0049093086, 7.325542e-09]


prediction_trajectory_error_mean - toy-problem-uqEnsemble3: [4.360629, 1.5691643, 0.76028776, 0.48939723, 0.390645, 0.2867098, 0.19949137, 0.12758735, 0.07674107, 0.03293396, 2.0120788e-07]
prediction_trajectory_error_mean - toy-problem-uqEst: [4.1037917, 2.35569, 0.7389877, 0.46809733, 0.40310052, 0.30174282, 0.21641865, 0.1354103, 0.08355798, 0.03521477, 4.615199e-08]


prediction_uncertainty_mean - toy-problem-uqEnsemble3: [0.4697134, 0.17677371, 0.12599078, 0.053375892, 0.03806868, 0.024076393, 0.011772508, 0.005354796, 0.0027467182, 0.000935216, 5.562895e-15]
prediction_uncertainty_mean - toy-problem-uqEst: [0.5597277, 0.5610771, 0.5576836, 0.5600485, 0.56045485, 0.56048256, 0.5600548, 0.5602634, 0.5598551, 0.55964965, 0.5592296]


prediction_uncertainty_gap_mean - toy-problem-uqEnsemble3: [-0.07536523, -0.01937183, 0.030954806, -0.0077987574, -0.010761946, -0.011762329, -0.013163914, -0.010593623, -0.006845916, -0.0031815288, -2.5150982e-08]
prediction_uncertainty_gap_mean - toy-problem-uqEst: [0.046753723, 0.2666158, 0.46531016, 0.50153637, 0.5100672, 0.5227647, 0.53300244, 0.5433371, 0.54941034, 0.5552478, 0.5592296]


prediction_uncertainty_gap_min - toy-problem-uqEnsemble3: [-3.200293, -1.4495728, -0.82090014, -0.68071085, -0.6092893, -0.5099707, -0.4359041, -0.3403778, -0.23777688, -0.11199872, -2.3980937e-07]
prediction_uncertainty_gap_min - toy-problem-uqEst: [-3.2680118, -1.6220052, -0.3853938, -0.19231199, -0.16395387, -0.12548575, -0.09443537, -0.059252262, -0.029860105, -0.003365001, 0.017708251]


prediction_uncertainty_gap_max - toy-problem-uqEnsemble3: [3.147433, 1.3851475, 1.184835, 0.5295608, 0.386682, 0.25939214, 0.14311223, 0.07270613, 0.03554647, 0.011089943, 1.3020501e-13]
prediction_uncertainty_gap_max - toy-problem-uqEst: [0.7765376, 0.7820884, 0.7894338, 0.78625923, 0.7889103, 0.7924885, 0.7976282, 0.7996886, 0.80333173, 0.80747354, 0.8139903]
